# Занятие 8, пара 2. Метрики: IoU и mAP своими руками


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Нашёл объект — это ещё надо проверить

В 08.1 мы договорились, что размечаем. Сегодня сравним с этой разметкой результаты детектора.
Детектор уже мог бы выдать рамки; обучение модели сейчас не требуется. Его ответы в примере заданы вручную,
чтобы было видно, **за какую ошибку меняется оценка**.

Пойдём по одной цепочке: насколько совпадают две рамки → какие объекты найдены →
сколько ложных срабатываний → как результат зависит от порога уверенности.
После пары вы сможете посчитать IoU, объяснить TP/FP/FN, precision/recall и смысл AP/mAP.
NumPy и Matplotlib достаточно, GPU и OpenCV не нужны. Основные определения и расчёты есть в этой версии тетради.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Пересечение полезно только вместе с размером объектов

Две рамки совпали на площади 20. Это много или мало? Для маленькой рамки — почти всё,
для большой — крохотный кусочек. Нужна относительная мера.

**IoU (Intersection over Union)** — площадь пересечения, делённая на площадь объединения.
Пересечение — область, попавшая в обе рамки. Объединение — всё, что попало хотя бы в одну.
Оно равно `площадь A + площадь B − пересечение`: общую часть нельзя посчитать дважды.
IoU=1 означает полное совпадение, IoU=0 — отсутствие пересечения положительной площади.

Используем углы `(x1,y1,x2,y2)` с положительной шириной и высотой. Границы непрерывные, без `+1`.
На картинке зелёная рамка — условный эталон, оранжевая — ответ. Сначала оцените совпадение на глаз.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

box_a = [0, 0, 6, 4]
box_b = [2, 1, 8, 5]
fig, ax = plt.subplots(figsize=(8, 5))
for box, color, label in zip([box_a, box_b], ['green', 'darkorange'], ['A', 'B']):
    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, alpha=0.25, color=color, label=label))
ax.set_xlim(-1, 9)
ax.set_ylim(6, -1)
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend()
plt.show()


Найдём пересечение по каждой оси отдельно. Левая граница общей области — **более правая**
из двух левых (`max`), правая — **более левая** из двух правых (`min`). Аналогично по y.
Здесь общая ширина 4, высота 3, площадь 12. Если рамки разошлись, разность может быть отрицательной,
поэтому ограничиваем её снизу нулём. Это не «исправление плохой рамки», а отсутствие пересечения.


In [ ]:
left = max(box_a[0], box_b[0])
top = max(box_a[1], box_b[1])
right = min(box_a[2], box_b[2])
bottom = min(box_a[3], box_b[3])
intersection_width = max(0, right - left)
intersection_height = max(0, bottom - top)
intersection = intersection_width * intersection_height
area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
union = area_a + area_b - intersection
print('Пересечение:', intersection, 'объединение:', union)
print('IoU:', intersection / union)


Обе площади равны 24, объединение 36, IoU=1/3. Деление только на площадь эталона дало бы 1/2 —
это уже другая мера, не IoU. Теперь соберём разобранные операции в функцию.
Нулевые и перевёрнутые рамки считаем ошибкой входа; отсутствие **предсказания** — это пустой список,
а не рамка нулевой площади.


In [ ]:
def box_iou(a, b):
    for box in (a, b):
        if box[2] <= box[0] or box[3] <= box[1]:
            raise ValueError('Нужна рамка положительной площади')
    left = max(a[0], b[0])
    top = max(a[1], b[1])
    right = min(a[2], b[2])
    bottom = min(a[3], b[3])
    intersection = max(0, right-left) * max(0, bottom-top)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    union = area_a + area_b - intersection
    return intersection / union

print('Совпали:', box_iou(box_a, box_a))
print('Разошлись:', box_iou(box_a, [10, 10, 12, 12]))


### ✏️ Ваш ход — Центр совпал — детекция хорошая?

Эталон `[0, 0, 10, 10]`. Придумайте `oversized_box` с **тем же центром**, но с IoU меньше 0.1.
Посчитайте IoU и объясните, почему расстояние между центрами не обнаруживает эту ошибку.
Координаты здесь на учебной плоскости, отрицательные допустимы. Это не рамка, которую нужно обрезать по фото.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.2.1', oversized_box)


**Мой вывод:** …


### 2. В большой сцене есть не только геометрия

Склеим две копии фотографии. Это **учебный коллаж**, а не две независимые фотографии для тестирования модели.
Зелёные рамки — две чашки по инструкции 08.1. Ложки сейчас не целевой класс.
Ответы детектора заданы нами: правильная рамка, повтор, ложная рамка на блюдце, вторая чашка.
Число `score` задаёт порядок уверенности детектора; его нельзя автоматически читать как вероятность правильности.

Для подписей используем `ax.annotate`: `xy` — точка, к которой ведёт стрелка, `xytext` — место текста, `arrowprops` задаёт вид стрелки. Разнесём подписи повторных рамок, иначе они перекроют друг друга.


In [ ]:
photo = np.load(DATA_DIR / 'segmentation' / 'coffee.npy')
scene = np.concatenate([photo, photo], axis=1)
truth = [[172, 18, 412, 310], [772, 18, 1012, 310]]
predictions = [
    {'score': 0.95, 'box': [177, 23, 417, 315]},
    {'score': 0.90, 'box': [170, 16, 410, 308]},
    {'score': 0.80, 'box': [65, 270, 195, 390]},
    {'score': 0.70, 'box': [777, 23, 1017, 315]},
]
fig, ax = plt.subplots(figsize=(16, 6))
ax.imshow(scene)
for box in truth:
    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='lime',linewidth=3))
label_positions = [(100, -35), (430, -35), (20, 435), (850, -35)]
for index, prediction in enumerate(predictions):
    x1, y1, x2, y2 = prediction['box']
    ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='darkorange',linewidth=2,linestyle='--'))
    ax.annotate(f'P{index+1}: {prediction["score"]}', xy=(x1, y1),
                xytext=label_positions[index], fontsize=12, backgroundcolor='white',
                arrowprops={'arrowstyle': '->', 'color': 'darkorange'})
ax.set_ylim(460, -65)
ax.set_title('Зелёные — эталоны; оранжевые — предоставленные ответы детектора')
plt.show()


### 3. Один объект нельзя найти дважды

Считаем для **одного изображения и одного класса**. Для разных классов и изображений сопоставление раздельное.
Установим порог IoU=0.5: рамка должна достаточно совпасть с эталоном. Это условие оценки,
а не порог яркости и не порог score.

Правило сопоставления:
1. Берём ответы по убыванию score.
2. Для текущего ответа ищем лучший ещё свободный эталон.
3. Если IoU не меньше порога, засчитываем TP и занимаем эталон.
4. Иначе это FP. Оставшиеся свободные эталоны — FN.

**TP** — верное обнаружение; **FP** — ложное срабатывание, включая лишнюю рамку на уже найденном объекте;
**FN** — пропущенный объект. Проверка «есть ли хоть одно хорошее IoU?» без учёта занятого эталона
ошибочно наградит детектор за сто одинаковых рамок.

Сначала сравним P1 с каждой чашкой. `np.argmax` возвращает индекс наибольшего числа.


In [ ]:
first_overlaps = []
for target in truth:
    first_overlaps.append(box_iou(predictions[0]['box'], target))
best_index = int(np.argmax(first_overlaps))
print('IoU P1 с чашками:', first_overlaps)
print('Лучший эталон:', best_index, 'порог пройден:', first_overlaps[best_index] >= 0.5)


P1 занял первую чашку. P2 почти идеален геометрически, но повторяет **тот же** объект.
Нельзя занять его второй раз. У P3 нет подходящей чашки; P4 находит вторую.

Теперь повторим эти действия в цикле. Множество `used` хранит занятые индексы.
`sorted(..., key=..., reverse=True)` сортирует ответы по score по убыванию.
Внутри перебора сначала пропускаем занятый эталон, затем запоминаем наибольший IoU.
Сохраним не только TP/FP, но и выбранный эталон: так можно проверить решение алгоритма глазами.
Для равных score сохраняется порядок исходного списка; при реальной оценке это соглашение тоже фиксируют.


In [ ]:
def match_boxes(predictions, truth, threshold=0.5):
    ordered = sorted(predictions, key=lambda item: item['score'], reverse=True)
    used = set()
    decisions = []
    for prediction in ordered:
        best_index = None
        best_overlap = -1.0
        for index, target in enumerate(truth):
            if index in used:
                continue
            overlap = box_iou(prediction['box'], target)
            if overlap > best_overlap:
                best_overlap = overlap
                best_index = index
        is_tp = best_index is not None and best_overlap >= threshold
        if is_tp:
            used.add(best_index)
        decisions.append({'score': prediction['score'], 'tp': int(is_tp),
                          'matched': best_index if is_tp else None})
    missed = len(truth) - len(used)
    return decisions, missed

decisions, missed = match_boxes(predictions, truth)
for row in decisions:
    print(row)
print('Пропущено:', missed)


### 4. Две доли отвечают на два разных вопроса

Из четырёх ответов два верные. Из двух настоящих чашек нашли обе.

- **Precision = TP / (TP + FP):** какая доля наших срабатываний правильна?
- **Recall = TP / (TP + FN):** какую долю настоящих объектов мы нашли?

По-русски их часто называют точностью и полнотой. Не путайте precision с accuracy классификации.
В детекции фон не разбит на конечный список «правильных отсутствий», поэтому привычный подсчёт TN
по всем возможным рамкам не имеет практического смысла. Общую accuracy здесь не считаем.


In [ ]:
tp = sum(row['tp'] for row in decisions)
fp = len(decisions) - tp
fn = missed
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print('TP, FP, FN:', tp, fp, fn)
print('Precision:', precision, 'recall:', recall)


### ✏️ Ваш ход — Три рамки на одну чашку

На кадре две чашки. Детектор выдал три одинаковые идеальные рамки первой чашки,
а вторую не нашёл. Не копируйте предыдущий список: сначала предскажите результат словами.
Запишите `duplicate_case = {'tp': ..., 'fp': ..., 'fn': ..., 'precision': ..., 'recall': ...}`.
Объясните, почему среднее IoU всех трёх рамок с ближайшим эталоном равно 1,
но качество обнаружения при этом далеко не идеальное.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.2.2', duplicate_case)


**Мой вывод:** …


### 5. Порог score двигает нас по списку ответов

Оставим только ответы score≥0.9: найдём первую чашку и её дубль, вторую потеряем.
Если понизить порог, появятся и полезные, и ложные ответы. **Порог score** выбирает предложения,
а **порог IoU** определяет, какие из выбранных считать достаточно точными. Это две разные ручки настройки.

Поскольку список уже отсортирован, можно считать результат после первых 1, 2, 3, 4 ответов.
Каждая новая строка увеличивает число выданных ответов. Recall растёт только при новом TP;
precision может и расти, и падать. Начнём с накопления TP обычным циклом.


In [ ]:
tp_flags = [row['tp'] for row in decisions]
running_tp = 0
precision_points = []
recall_points = []
for rank, flag in enumerate(tp_flags, start=1):
    running_tp += flag
    current_precision = running_tp / rank
    current_recall = running_tp / len(truth)
    precision_points.append(current_precision)
    recall_points.append(current_recall)
    print('Ответов:', rank, 'TP:', running_tp,
          'precision:', round(current_precision, 3), 'recall:', current_recall)


Получили строки: `(recall, precision)` = (0.5,1), (0.5,0.5), (0.5,1/3), (1,0.5).
Нарисуем их точками: две оси — **доли**, не координаты изображения.
Одинаковая полнота при падающей точности означает, что добавились ложные ответы.
Не соединяем их произвольной гладкой кривой: между этими порогами новых наблюдений у нас нет.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(recall_points, precision_points, s=80)
for rank, (r, p) in enumerate(zip(recall_points, precision_points), start=1):
    ax.annotate(str(rank), (r, p), xytext=(7, 7), textcoords='offset points')
ax.set_xlim(0, 1.1)
ax.set_ylim(0, 1.1)
ax.set_xlabel('Recall — доля найденных объектов')
ax.set_ylabel('Precision — доля верных срабатываний')
ax.set_title('Число рядом с точкой — сколько ответов оставили')
ax.grid(alpha=0.3)
plt.show()


### 6. AP: хотим хорошие ответы раньше плохих

Одна точка precision/recall зависит от выбранного порога score. **Average Precision (AP)**
обобщает ранжированный список. Для понимания используем учебный **неинтерполированный AP**:
при каждом новом TP добавляем `прирост recall × precision на этом шаге`.
FP не увеличивает recall, но снижает precision последующих TP. Ненайденные объекты дают нулевой вклад.

В нашем примере первый TP добавил 0.5×1, второй — 0.5×0.5. Итого 0.75.
При двух эталонах это также среднее precision на позициях TP: (1 + 0.5)/2.
Если один эталон не найден, делитель всё равно **2**, а не число найденных.
Мы не используем площадь трапеций через точки: это другой способ суммирования.


In [ ]:
ap_example = 0.5 * 1.0 + 0.5 * 0.5
print('Учебный AP:', ap_example)


Соберём уже разобранные шаги. Вход `flags` — список 1/0 **после корректного сопоставления**
по убыванию score, `n_truth` — число эталонных объектов этого класса.
Если в оценочном наборе нет объектов класса, вернём `None`: качество поиска такого класса
нельзя оценить как обычный AP. FP на отрицательных изображениях при этом учитываются,
если класс есть на других изображениях общей оценочной выборки.


In [ ]:
def average_precision(flags, n_truth):
    if n_truth == 0:
        return None
    if sum(flags) > n_truth:
        raise ValueError('TP больше эталонов: проверьте сопоставление')
    running_tp = 0
    total = 0.0
    for rank, flag in enumerate(flags, start=1):
        running_tp += flag
        if flag == 1:
            precision_here = running_tp / rank
            recall_gain = 1 / n_truth
            total += recall_gain * precision_here
    return total

print(average_precision(tp_flags, len(truth)))


### ✏️ Ваш ход — Одинаковый финал, разный порядок

Есть два настоящих объекта и четыре ответа: два TP и два FP.
Создайте `best_order` и `worst_order` из двух единиц и двух нулей так, чтобы учебный AP был
соответственно максимальным и минимальным. Посчитайте оба результата.
Итоговые precision и recall одинаковы — почему AP различается? Как это связано с работой при высоком пороге score?
Считаем, что сопоставление выполнено корректно и речь о двух разных найденных объектах.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.2.3', best_order, worst_order)


**Мой вывод:** …


### 7. Почему рядом с mAP надо читать мелкий шрифт

Для нескольких классов сначала считают AP каждого класса, затем среднее — **mAP**.
В учебном примере AP чашек 0.75, AP ложек 0.5 → mAP 0.625; это не среднее IoU рамок.
Классы имеют одинаковый вес в этом среднем, даже если объектов одного класса больше.

**AP50** фиксирует порог совпадения IoU=0.50. Более строгий порог, например 0.75, сильнее штрафует
неточные рамки. Стандартная сводная COCO AP усредняет результаты для 10 порогов IoU от 0.50 до 0.95
с шагом 0.05 и использует интерполированную precision на 101 уровне recall; учитываются также
ограничения количества детекций и специальные случаи разметки.

Наша функция объясняет механизм ранжирования, **она не является реализацией COCO AP**.
Даже на тех же рамках итог может немного отличаться из-за способа интерполяции/суммирования.
Для сравнения с опубликованной моделью нужен её официальный протокол, общий датасет и реализация evaluator.
Источник точного протокола: [COCOeval](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py).
На тестовой части нельзя многократно подбирать удобный порог: его выбирают на validation.


In [ ]:
class_ap = {'cup': 0.75, 'spoon': 0.5}
print('Учебный mAP:', sum(class_ap.values()) / len(class_ap))
for threshold in [0.5, 0.95]:
    result, missed_at_threshold = match_boxes(predictions, truth, threshold)
    flags = [row['tp'] for row in result]
    print('IoU-порог:', threshold, 'TP:', sum(flags),
          'FN:', missed_at_threshold, 'учебный AP:', average_precision(flags, len(truth)))


## Что теперь умеем объяснить

IoU сравнивает геометрию двух объектов. Сопоставление решает, какой объект уже найден.
Precision/recall описывают два вида ошибки, AP — порядок предложений при изменении порога.
Число без названия протокола и условий сравнения мало что говорит.

В проекте **классификации** из ЛР4 рамки и mAP обязательными не становятся. Там будем проверять метки,
групповое разбиение и ошибки классификатора. Сегодняшний общий навык — сначала определить,
какую ошибку измеряет метрика, и построить контрпример, который она может не заметить.

### Дополнительно — маски и маленькие объекты, 10–15 минут

Для бинарных масок пересечение — `mask_a & mask_b`, объединение — `mask_a | mask_b`.
`np.count_nonzero` считает True, то есть площадь в пикселях. Если обе маски пусты, надо заранее выбрать соглашение:
здесь вернём None и не включим такой пример в среднее без отдельного обсуждения.
Для непустой истинной маски и пустого ответа IoU=0.

На большом изображении можно незаметно сдвинуть маленький объект на 2 пикселя и сильно снизить IoU.
Сравните квадрат 10×10 и 100×100 при одинаковом сдвиге. Почему «всего два пикселя» — не универсальное оправдание?


In [ ]:
def mask_iou(a, b):
    intersection = np.count_nonzero(a & b)
    union = np.count_nonzero(a | b)
    if union == 0:
        return None
    return intersection / union

for side in [10, 100]:
    original = [0, 0, side, side]
    shifted = [2, 0, side+2, side]
    print('Сторона:', side, 'IoU:', box_iou(original, shifted))
